<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Web/blob/main/171-Vectorstores-embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectorstores and Embeddings

Recall the overall workflow for retrieval augmented generation (RAG):

In [1]:
!pip install --upgrade --force-reinstall openai
!pip install --upgrade --force-reinstall langchain langchain-community langchain-text-splitters langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.9/124.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3

ERROR: Operation cancelled by user
^C


In [5]:
import sys
!{sys.executable} -m pip install langchain-community

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.3.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 2.3.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.43.0 which is incompatible.


In [1]:
!pip install pypdf
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [16]:
from google.colab import userdata
import openai
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
openai.api_key  = os.environ["OPENAI_API_KEY"]

We just discussed `Document Loading` and `Splitting`.

In [17]:
# Colab 셀에서 GitHub URL의 PDF 파일을 직접 다운로드하는 코드입니다.

import requests

url = "https://github.com/JSJeong-me/GPT-Web/raw/main/images/MachineLearning-Lecture01.pdf"
filename = "MachineLearning-Lecture01.pdf"

response = requests.get(url)
if response.status_code == 200:
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"{filename} 다운로드 완료")
else:
    print("다운로드 실패, 상태 코드:", response.status_code)

MachineLearning-Lecture01.pdf 다운로드 완료


In [18]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    # Duplicate documents on purpose - messy data
    PyPDFLoader("MachineLearning-Lecture01.pdf"),
    # PyPDFLoader("MachineLearning-Lecture01.pdf"),
    # PyPDFLoader("MachineLearning-Lecture02.pdf"),
    # PyPDFLoader("MachineLearning-Lecture03.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [19]:


# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

In [20]:
splits = text_splitter.split_documents(docs)

In [21]:
len(splits)

57

## Embeddings

Let's take our splits and embed them.

In [22]:
from langchain_openai import OpenAIEmbeddings
embedding = OpenAIEmbeddings()

In [23]:
import sys
!{sys.executable} -m pip install langchain-openai

In [24]:
sentence1 = "i like dogs"
sentence2 = "i like canines"
sentence3 = "the weather is ugly outside"

In [25]:
embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

In [26]:
import numpy as np

In [27]:
np.dot(embedding1, embedding2)

np.float64(0.9631511809630343)

In [28]:
np.dot(embedding1, embedding3)

np.float64(0.7702031371038218)

In [29]:
np.dot(embedding2, embedding3)

np.float64(0.759054062979165)

## Vectorstores

In [30]:
from langchain_community.vectorstores import Chroma

In [31]:
persist_directory = 'docs/chroma/'

In [32]:
!rm -rf ./docs/chroma  # remove old database files if any

In [33]:
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_directory
)

In [34]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    # Duplicate documents on purpose - messy data
    PyPDFLoader("MachineLearning-Lecture01.pdf"),
    # PyPDFLoader("MachineLearning-Lecture01.pdf"),
    # PyPDFLoader("MachineLearning-Lecture02.pdf"),
    # PyPDFLoader("MachineLearning-Lecture03.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [35]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

In [36]:
splits = text_splitter.split_documents(docs)

In [37]:
print(vectordb._collection.count())

57


### Similarity Search

In [38]:
question = "is there an email i can ask for help"

In [39]:
docs = vectordb.similarity_search(question,k=3)

In [40]:
len(docs)

3

In [41]:
docs[0].page_content

"cs229-qa@cs.stanford.edu. This goes to an account that's read by all the TAs and me. So \nrather than sending us email individually, if you send email to this account, it will \nactually let us get back to you maximally quickly with answers to your questions.  \nIf you're asking questions about homework problems, please say in the subject line which \nassignment and which question the email refers to, since that will also help us to route \nyour question to the appropriate TA or to me appropriately and get the response back to \nyou quickly.  \nLet's see. Skipping ahead — let's see — for homework, one midterm, one open and term \nproject. Notice on the honor code. So one thing that I think will help you to succeed and \ndo well in this class and even help you to enjoy this class more is if you form a study \ngroup.  \nSo start looking around where you're sitting now or at the end of class today, mingle a \nlittle bit and get to know your classmates. I strongly encourage you to form st

Let's save this so we can use it later!

In [42]:
vectordb.persist()

/tmp/ipykernel_2846/3711397106.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


## Failure modes

This seems great, and basic similarity search will get you 80% of the way there very easily.

But there are some failure modes that can creep up.

Here are some edge cases that can arise - we'll fix them in the next class.

In [43]:
question = "what did they say about matlab?"

In [44]:
docs = vectordb.similarity_search(question,k=5)

Notice that we're getting duplicate chunks (because of the duplicate `MachineLearning-Lecture01.pdf` in the index).

Semantic search fetches all similar documents, but does not enforce diversity.

`docs[0]` and `docs[1]` are indentical.

In [45]:
docs[0]

Document(metadata={'source': 'MachineLearning-Lecture01.pdf', 'page_label': '9', 'author': '', 'creationdate': '2008-07-11T11:25:23-07:00', 'page': 8, 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'moddate': '2008-07-11T11:25:23-07:00', 'total_pages': 22, 'title': ''}, page_content='those homeworks will be done in either MATLAB or in Octave, which is sort of — I \nknow some people call it a free version of MATLAB, which it sort of is, sort of isn\'t.  \nSo I guess for those of you that haven\'t seen MATLAB before, and I know most of you \nhave, MATLAB is I guess part of the programming language that makes it very easy to \nwrite codes using matrices, to write code for numerical routines, to move data around, to \nplot data. And it\'s sort of an extremely easy to learn tool to use for implementing a lot of \nlearning algorithms.  \nAnd in case some of you want to work on your own home computer or something if you \ndon\'t have a MATLAB license

In [46]:
docs[1]

Document(metadata={'page': 8, 'page_label': '9', 'author': '', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationdate': '2008-07-11T11:25:23-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:23-07:00', 'source': 'MachineLearning-Lecture01.pdf', 'title': '', 'total_pages': 22}, page_content='into his office and he said, "Oh, professor, professor, thank you so much for your \nmachine learning class. I learned so much from it. There\'s this stuff that I learned in your \nclass, and I now use every day. And it\'s helped me make lots of money, and here\'s a \npicture of my big house."  \nSo my friend was very excited. He said, "Wow. That\'s great. I\'m glad to hear this \nmachine learning stuff was actually useful. So what was it that you learned? Was it \nlogistic regression? Was it the PCA? Was it the data networks? What was it that you \nlearned that was so helpful?" And the student said, "Oh, it was the MATLAB."  \nSo for those of you that don\'t know

We can see a new failure mode.

The question below asks a question about the third lecture, but includes results from other lectures as well.

In [47]:
question = "what did they say about regression in the third lecture?"

In [48]:
docs = vectordb.similarity_search(question,k=5)

In [49]:
for doc in docs:
    print(doc.metadata)

{'author': '', 'page': 8, 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2008-07-11T11:25:23-07:00', 'title': '', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'MachineLearning-Lecture01.pdf', 'moddate': '2008-07-11T11:25:23-07:00', 'page_label': '9', 'total_pages': 22}
{'creator': 'PScript5.dll Version 5.2.2', 'title': '', 'page': 8, 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'MachineLearning-Lecture01.pdf', 'page_label': '9', 'author': '', 'total_pages': 22, 'moddate': '2008-07-11T11:25:23-07:00', 'creationdate': '2008-07-11T11:25:23-07:00'}
{'author': '', 'title': '', 'total_pages': 22, 'creationdate': '2008-07-11T11:25:23-07:00', 'moddate': '2008-07-11T11:25:23-07:00', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'page': 9, 'source': 'MachineLearning-Lecture01.pdf', 'creator': 'PScript5.dll Version 5.2.2', 'page_label': '10'}
{'title': '', 'moddate': '2008-07-11T11:25:23-07:00', 'total_pages': 22, 'page_label': '14', 'producer': 'Acrobat 

In [50]:
print(docs[4].page_content)

But if you don't quite know or if you're not quite sure, that's fine, too. We'll go over it in 
the review sections.  
So there are a couple more logistical things I should deal with in this class. One is that, as 
most of you know, CS229 is a televised class. And in fact, I guess many of you are 
probably watching this at home on TV, so I'm gonna say hi to our home viewers.  
So earlier this year, I approached SCPD, which televises these classes, about trying to 
make a small number of Stanford classes publicly available or posting the videos on the 
web. And so this year, Stanford is actually starting a small pilot program in which we'll 
post videos of a small number of classes online, so on the Internet in a way that makes it 
publicly accessible to everyone. I'm very excited about that because machine learning in 
school, let's get the word out there.  
One of the consequences of this is that — let's see — so videos  or pictures of the students 
in this classroom will not be poste

Approaches discussed in the next lecture can be used to address both!